# Frozen full-AIT valence directions on SST, IMDb, and DynaSent

This notebook loads the **AIT-validation-selected layer and saved direction checkpoint** for mean difference and one-dimensional DAS from full-AIT run `2026-09-23_09-10_CDT`. It evaluates those frozen choices on SST, IMDb, DynaSent R1, and DynaSent R2 for GPT-2 Small and Qwen3-0.6B Base. Logistic regression is intentionally excluded because it performed poorly in the full-AIT experiment.

No direction is refit and no layer, checkpoint, method, or intervention strength is selected from the four transfer datasets. Mean difference retains final-token patching; DAS retains the all-token layer-selection/evaluation protocol used by the full-AIT experiment.

## Before running

1. Use a Colab GPU runtime.
2. Confirm the completed source run exists under `MyDrive/sentiment-geometry/full-ait-last-token-directions/runs`.
3. Provide a Hugging Face token with read access to the private SST, IMDb, and DynaSent repositories when prompted. The token is removed after evaluation.
4. Leave the pinned model and dataset revisions unchanged for a reportable run.
5. Run GPT-2 first with `RUN_STAGE = "gpt2"` and `RESUME_RUN_ID = None`. Copy the printed run ID after it finishes.
6. In a fresh runtime, set `RUN_STAGE = "qwen"` and `RESUME_RUN_ID` to that GPT-2 run ID. The notebook will preserve/reuse GPT-2's result files and evaluate only Qwen.
7. Tune `GPT2_BATCH_SIZE` and `QWEN_BATCH_SIZE` independently for the L4. Changing evaluation batch size does not change the frozen scientific protocol.

In [1]:
# --------------------------- User settings ---------------------------
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Optional commit or tag. Existing checkouts must match it.
CONFIG_RELATIVE_PATH = "configs/ait_valence_transfer_evaluation.yaml"

DRIVE_STORAGE_ROOT = "/content/drive/MyDrive/sentiment-geometry"
SOURCE_RUN_ID = "2026-09-23_09-10_CDT"
TIMEZONE_NAME = "America/Chicago"
RUN_STAGE = "both"  # A100 default; alternatives are "gpt2" or "qwen".
RESUME_RUN_ID = None  # Keep None for a fresh combined run.

DEVICE = "cuda"
DTYPE = "auto"
ALL_MODEL_NAMES = ["gpt2-small", "qwen-0.6b"]
GPT2_BATCH_SIZE = 16
QWEN_BATCH_SIZE = 8
RUN_STAGES = {
    "both": {
        "evaluate": ["gpt2-small", "qwen-0.6b"],
        "reuse": [],
        "batch_sizes": {
            "gpt2-small": GPT2_BATCH_SIZE,
            "qwen-0.6b": QWEN_BATCH_SIZE,
        },
    },
    "gpt2": {
        "evaluate": ["gpt2-small"],
        "reuse": [],
        "batch_sizes": {"gpt2-small": GPT2_BATCH_SIZE},
    },
    "qwen": {
        "evaluate": ["qwen-0.6b"],
        "reuse": ["gpt2-small"],
        "batch_sizes": {"qwen-0.6b": QWEN_BATCH_SIZE},
    },
}
if RUN_STAGE not in RUN_STAGES:
    raise ValueError(f"RUN_STAGE must be one of {sorted(RUN_STAGES)}.")
ACTIVE_STAGE = RUN_STAGES[RUN_STAGE]
EVALUATION_MODEL_NAMES = list(ACTIVE_STAGE["evaluate"])
REUSE_COMPLETED_MODEL_NAMES = list(ACTIVE_STAGE["reuse"])
MODEL_BATCH_SIZES = dict(ACTIVE_STAGE["batch_sizes"])
RESULT_MODEL_NAMES = REUSE_COMPLETED_MODEL_NAMES + EVALUATION_MODEL_NAMES
RUN_EVALUATION = True

## 1. Install and verify the project

The notebook uses the reusable AIT transfer configuration and frozen-direction evaluation APIs. Dataset loading, artifact validation, patching, metrics, and persistence remain in `sentiment_geometry`.

In [2]:
import importlib
import os
import pkgutil
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")

project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if PROJECT_REVISION is not None:
    expected_commit = subprocess.check_output(
        ["git", "rev-parse", PROJECT_REVISION], cwd=PROJECT_ROOT, text=True
    ).strip()
    if project_commit != expected_commit:
        raise RuntimeError(
            f"Existing checkout is {project_commit}, expected {expected_commit}. "
            "Use a fresh runtime or update PROJECT_REVISION."
        )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)
project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)
for loaded_name in list(sys.modules):
    if loaded_name == "sentiment_geometry" or loaded_name.startswith("sentiment_geometry."):
        del sys.modules[loaded_name]
importlib.invalidate_caches()

import sentiment_geometry
expected_package_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
imported_package_root = Path(sentiment_geometry.__file__).resolve().parent
if imported_package_root != expected_package_root:
    raise ImportError(
        f"Imported sentiment_geometry from {imported_package_root}, "
        f"expected {expected_package_root}. Restart the runtime and rerun from the top."
    )
module_names = sorted(
    module.name
    for module in pkgutil.walk_packages(sentiment_geometry.__path__, prefix="sentiment_geometry.")
    if module.name != "sentiment_geometry.__main__"
)
for module_name in module_names:
    importlib.import_module(module_name)
required_apis = {
    "sentiment_geometry.experiments": {
        "AITValenceTransferConfig",
        "load_frozen_direction_selections",
        "run_frozen_sentiment_direction_evaluation",
    },
    "sentiment_geometry.persistence": {
        "RunArtifactStore", "maybe_mount_google_drive", "prepare_timestamped_run"
    },
}
for module_name, api_names in required_apis.items():
    module = importlib.import_module(module_name)
    missing_apis = sorted(name for name in api_names if not hasattr(module, name))
    if missing_apis:
        raise ImportError(f"{module_name} is missing {missing_apis}.")
print("Project commit:", project_commit)
print("Imported package from:", imported_package_root)

Project commit: f86917cfb76c6c336856323868741f1b2dd0f96f
Imported package from: /content/sentiment-manifold/sentiment_geometry


## 2. Mount Drive and configure the locked evaluation

The YAML pins both model revisions, all four directed-pair dataset revisions, the source run, validation-selection provenance, locked AIT-test provenance, and method-specific patch positions. The GPT-2 stage creates a minute-stamped output run. The Qwen stage must explicitly resume that same run and reuse its completed GPT-2 files.

In [3]:
import json

import pandas as pd
import torch
from IPython.display import display

from sentiment_geometry.experiments import AITValenceTransferConfig
from sentiment_geometry.persistence import (
    RunArtifactStore, maybe_mount_google_drive, prepare_timestamped_run
)

if RUN_EVALUATION and DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime before continuing.")
maybe_mount_google_drive(True)
plan = AITValenceTransferConfig.load(PROJECT_ROOT / CONFIG_RELATIVE_PATH)
if [model.name for model in plan.models] != ALL_MODEL_NAMES:
    raise RuntimeError("The pinned model list changed unexpectedly.")
if set(MODEL_BATCH_SIZES) != set(EVALUATION_MODEL_NAMES):
    raise RuntimeError("MODEL_BATCH_SIZES must specify only evaluated models.")
if RUN_STAGE == "gpt2" and RESUME_RUN_ID is not None:
    print("Resuming/restarting the GPT-2 stage in the requested existing run.")
if RUN_STAGE == "qwen" and RESUME_RUN_ID is None:
    raise RuntimeError(
        "The Qwen stage must set RESUME_RUN_ID to the completed GPT-2 run ID."
    )

SOURCE_RUN_ROOT = plan.source_run_root(
    DRIVE_STORAGE_ROOT, source_run_id=SOURCE_RUN_ID
)
SOURCE_MANIFEST_PATH = SOURCE_RUN_ROOT / "run_manifest.json"
if not SOURCE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(SOURCE_MANIFEST_PATH)
source_manifest = json.loads(SOURCE_MANIFEST_PATH.read_text(encoding="utf-8"))
if (
    plan.require_completed_source_status
    and source_manifest.get("status") != "completed"
):
    raise RuntimeError(
        f"Source run {SOURCE_RUN_ID} is not completed: {source_manifest.get('status')!r}"
    )

if "RUN_LAYOUT" not in globals() or RESUME_RUN_ID is not None:
    RUN_LAYOUT = prepare_timestamped_run(
        DRIVE_STORAGE_ROOT,
        experiment_name=plan.output_experiment_name,
        timezone_name=TIMEZONE_NAME,
        resume_run_id=RESUME_RUN_ID,
    )
config = plan.build_evaluation_config(
    storage_root=DRIVE_STORAGE_ROOT,
    source_run_id=SOURCE_RUN_ID,
    output_dir=RUN_LAYOUT.results_dir,
    evaluated_model_names=EVALUATION_MODEL_NAMES,
    reuse_completed_models=REUSE_COMPLETED_MODEL_NAMES,
    device=DEVICE,
    dtype=DTYPE,
    batch_sizes=MODEL_BATCH_SIZES,
)
METHOD_ORDER = list(plan.methods)
DATASET_ORDER = [dataset.name for dataset in plan.datasets]
RunArtifactStore(RUN_LAYOUT.root).write_json("requested_config.json", config.to_dict())
RUN_LAYOUT.update_manifest(
    status="configured",
    metadata={
        "project_commit": project_commit,
        "source_run_id": SOURCE_RUN_ID,
        "source_run_status": source_manifest.get("status"),
        "run_stage": RUN_STAGE,
        "selection_dataset": plan.selection_dataset,
        "source_evaluation_dataset": plan.source_evaluation_dataset,
        "evaluated_models": EVALUATION_MODEL_NAMES,
        "reused_completed_models": REUSE_COMPLETED_MODEL_NAMES,
        "evaluation_datasets": DATASET_ORDER,
        "method_patch_positions": plan.method_patch_positions,
        "configuration": "requested_config.json",
    },
)
display(pd.DataFrame([
    {
        "dataset": dataset.name, "repository": dataset.repo_id,
        "revision": dataset.revision, "split": dataset.split
    }
    for dataset in plan.datasets
]))
print("Parent run:  ", SOURCE_RUN_ROOT)
print("New run ID:  ", RUN_LAYOUT.run_id)
print("New run root:", RUN_LAYOUT.root)
print("Stage:       ", RUN_STAGE)
print("Evaluating:  ", EVALUATION_MODEL_NAMES)
print("Reusing:     ", REUSE_COMPLETED_MODEL_NAMES)
print("Batch sizes: ", MODEL_BATCH_SIZES)

Mounted at /content/drive


,dataset,repository,revision,split
0,sst,kokolamba/sentiment-manifold-sst-pythia-2.8b,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,test
1,imdb,kokolamba/sentiment-manifold-imdb-pythia-2.8b,06586d20342aa46fa61525b1f7609ab065983e3e,test
2,dynasent_r1,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,test
3,dynasent_r2,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,test


Parent run:   /content/drive/MyDrive/sentiment-geometry/full-ait-last-token-directions/runs/2026-09-23_09-10_CDT
New run ID:   2026-09-24_11-20_CDT
New run root: /content/drive/MyDrive/sentiment-geometry/ait-valence-ood-evaluation/runs/2026-09-24_11-20_CDT
Stage:        both
Evaluating:   ['gpt2-small', 'qwen-0.6b']
Reusing:      []
Batch sizes:  {'gpt2-small': 16, 'qwen-0.6b': 8}


## 3. Authenticate to the private Hugging Face datasets

The token is hidden, used only while resolving/loading the private datasets, and then removed from the process and notebook cache.

In [4]:
import gc
from getpass import getpass

from huggingface_hub import HfApi
from huggingface_hub.utils import reset_sessions

_RUNTIME_SECRETS = {}

def get_runtime_secret(name):
    if name not in _RUNTIME_SECRETS:
        value = getpass(f"Enter {name} (input hidden): " ).strip()
        if not value:
            raise RuntimeError(f"{name} was not provided.")
        _RUNTIME_SECRETS[name] = value
    return _RUNTIME_SECRETS[name]

def clear_hf_credentials(env_name="HF_TOKEN"):
    os.environ.pop(env_name, None)
    value = _RUNTIME_SECRETS.pop("HF_TOKEN", None)
    if value is not None:
        del value
    reset_sessions()
    gc.collect()

_token = get_runtime_secret("HF_TOKEN")
try:
    hf_account = HfApi(token=_token).whoami()["name"]
except BaseException:
    clear_hf_credentials()
    raise
finally:
    del _token
print(f"Authenticated to Hugging Face as {hf_account}. Token value was not displayed.")

Authenticated to Hugging Face as kokolamba. Token value was not displayed.


## 4. Load the frozen full-AIT best layers

This reads one validation-selected layer and its checkpoint for each model available in the current stage. It also verifies that the source run's locked `ait_test` result used that same layer. Each displayed cell is `Lxx (AIT-validation logit-flip %)`.

In [5]:
from sentiment_geometry.experiments import load_frozen_direction_selections

MODEL_LABELS = {
    "gpt2-small": "GPT-2 Small",
    "qwen-0.6b": "Qwen3-0.6B Base",
}
METHOD_LABELS = {
    "mean_diff": "Mean Difference",
    "das": "DAS (1D)",
}
selection_rows = []
for model in plan.models:
    if model.name not in RESULT_MODEL_NAMES:
        continue
    for selected in load_frozen_direction_selections(config, model):
        selection_rows.append({
            "model": selected.model, "method": selected.method,
            "fit_position": selected.fit_position,
            "selected_layer": selected.selected_layer,
            "selection_dataset": selected.selection_dataset,
            "selection_metric": selected.selection_metric,
            "selection_value_percent": selected.selection_value_percent,
            "patch_position": plan.method_patch_positions[selected.method],
            "source_direction_checkpoint": str(selected.checkpoint_path),
        })
FROZEN_SELECTION = pd.DataFrame(selection_rows)
if len(FROZEN_SELECTION) != len(RESULT_MODEL_NAMES) * len(METHOD_ORDER):
    raise RuntimeError("The frozen full-AIT selection grid is incomplete.")
if set(FROZEN_SELECTION["selection_dataset"]) != {"ait_eval"}:
    raise RuntimeError("A layer was not selected on AIT validation.")
if set(FROZEN_SELECTION["selection_metric"]) != {"logit_flip_percent"}:
    raise RuntimeError("A layer was not selected by validation logit-flip percent.")
layer_display = FROZEN_SELECTION.copy()
layer_display["layer_and_score"] = layer_display.apply(
    lambda row: f"L{int(row['selected_layer']):02d} ({float(row['selection_value_percent']):.1f}%)",
    axis=1,
)
best_layer_table = layer_display.pivot(
    index="method", columns="model", values="layer_and_score"
).reindex(index=METHOD_ORDER, columns=RESULT_MODEL_NAMES)
best_layer_table.index = [METHOD_LABELS[method] for method in METHOD_ORDER]
best_layer_table.columns = [MODEL_LABELS[model] for model in RESULT_MODEL_NAMES]
display(best_layer_table.style.set_properties(**{"text-align": "center"}))
display(FROZEN_SELECTION[[
    "model", "method", "selected_layer", "patch_position",
    "selection_dataset", "selection_value_percent"
]].sort_values(["model", "method"]).reset_index(drop=True))

,GPT-2 Small,Qwen3-0.6B Base
Mean Difference,L11 (72.0%),L28 (98.9%)
DAS (1D),L11 (125.6%),L26 (112.4%)


,model,method,selected_layer,patch_position,selection_dataset,selection_value_percent
0,gpt2-small,das,11,all,ait_eval,125.609760
1,gpt2-small,mean_diff,11,final,ait_eval,71.951213
2,qwen-0.6b,das,26,all,ait_eval,112.365586
3,qwen-0.6b,mean_diff,28,final,ait_eval,98.924730


## 5. Run the locked transfer evaluations

The package writes aggregate metrics, per-case records, frozen-layer provenance, dataset summaries, direction metadata, and resolved configuration files to Drive.

In [6]:
from sentiment_geometry.experiments import run_frozen_sentiment_direction_evaluation

if not RUN_EVALUATION:
    print("Evaluation execution is disabled. Set RUN_EVALUATION = True.")
else:
    os.environ.pop(config.hf_token_env, None)
    try:
        os.environ[config.hf_token_env] = get_runtime_secret("HF_TOKEN")
        RUN_LAYOUT.update_manifest(status="running")
        completed_results_dir = run_frozen_sentiment_direction_evaluation(config)
        if completed_results_dir.resolve() != RUN_LAYOUT.results_dir.resolve():
            raise RuntimeError(f"Unexpected results directory: {completed_results_dir}")
        RUN_LAYOUT.update_manifest(status="evaluation-completed")
    except BaseException as error:
        RUN_LAYOUT.update_manifest(
            status="failed", metadata={"failure_type": type(error).__name__}
        )
        raise
    finally:
        clear_hf_credentials(config.hf_token_env)
    print("Evaluation results saved to:", completed_results_dir)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

r2_qwen_0_6b_directed_pairs/test-00000-o(…):   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

Evaluation results saved to: /content/drive/MyDrive/sentiment-geometry/ait-valence-ood-evaluation/runs/2026-09-24_11-20_CDT/results


## 6. Audit and display essential results

The audit requires the complete model × method × dataset grid for every model available in the current stage. The GPT-2 stage is marked partial; the resumed Qwen stage audits both models before marking the shared run completed.

In [7]:
from datetime import datetime
from itertools import product
from zoneinfo import ZoneInfo

METRICS_PATH = RUN_LAYOUT.results_dir / "all_models_metrics.csv"
SELECTION_PATH = RUN_LAYOUT.results_dir / "all_models_layer_selection.csv"
SUMMARY_PATH = RUN_LAYOUT.results_dir / "all_models_dataset_summary.csv"
for required_path in (METRICS_PATH, SELECTION_PATH, SUMMARY_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)
metrics = pd.read_csv(METRICS_PATH)
saved_selection = pd.read_csv(SELECTION_PATH)
dataset_summary = pd.read_csv(SUMMARY_PATH)
metric_keys = ["model", "method", "fit_position", "dataset"]
if metrics.duplicated(metric_keys).any():
    raise RuntimeError("Duplicate aggregate metric cells were saved.")
expected_cells = set(product(RESULT_MODEL_NAMES, METHOD_ORDER, ["final"], DATASET_ORDER))
actual_cells = set(metrics[metric_keys].itertuples(index=False, name=None))
if expected_cells != actual_cells:
    raise RuntimeError(
        f"Incomplete metric grid. Missing={sorted(expected_cells - actual_cells)}; "
        f"unexpected={sorted(actual_cells - expected_cells)}"
    )
for method, position in plan.method_patch_positions.items():
    observed = set(metrics.loc[metrics["method"] == method, "patch_position"])
    if observed != {position}:
        raise RuntimeError(f"Unexpected patch position for {method}: {observed}")
if set(saved_selection["selection_dataset"]) != {"ait_eval"}:
    raise RuntimeError("Saved layers were not selected exclusively on AIT validation.")
for model_name in RESULT_MODEL_NAMES:
    model_dir = RUN_LAYOUT.results_dir / model_name
    for filename in (
        "metrics.csv", "patching_records.csv", "layer_selection.csv",
        "dataset_summary.csv", "direction_metadata.csv", "resolved_config.json"
    ):
        if not (model_dir / filename).is_file():
            raise FileNotFoundError(model_dir / filename)
    patching_records = pd.read_csv(model_dir / "patching_records.csv")
    expected_records = int(
        dataset_summary.loc[dataset_summary["model"] == model_name, "n_directed_cases"].sum()
    ) * len(METHOD_ORDER)
    if len(patching_records) != expected_records:
        raise RuntimeError(
            f"{model_name} saved {len(patching_records)} per-case records; "
            f"expected {expected_records}."
        )
completed_at = datetime.now(ZoneInfo(RUN_LAYOUT.timezone_name)).isoformat(timespec="minutes")
FINAL_COMBINED_RUN = set(RESULT_MODEL_NAMES) == set(ALL_MODEL_NAMES)
completion_metadata = {
    "stage_completed_at": completed_at, "aggregate_metric_rows": len(metrics),
    "frozen_layer_rows": len(saved_selection),
    "dataset_summary_rows": len(dataset_summary),
    "completed_models": RESULT_MODEL_NAMES,
    "pending_models": sorted(set(ALL_MODEL_NAMES) - set(RESULT_MODEL_NAMES)),
}
if FINAL_COMBINED_RUN:
    completion_metadata["completed_at"] = completed_at
RUN_LAYOUT.update_manifest(
    status="completed" if FINAL_COMBINED_RUN else "partial",
    metadata=completion_metadata,
)
display(dataset_summary.sort_values(["model", "dataset"]).reset_index(drop=True))
print(f"Validated {len(metrics)} aggregate metric rows and all per-case records.")
if not FINAL_COMBINED_RUN:
    print("GPT-2 stage complete. Use this run ID for the Qwen stage:", RUN_LAYOUT.run_id)

,model,dataset,role,repo_id,config_name,split,requested_revision,resolved_revision,n_examples,n_directed_cases
0,gpt2-small,dynasent_r1,frozen_ood_evaluation,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,r1_gpt2_small_directed_pairs,test,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,80,80
1,gpt2-small,dynasent_r2,frozen_ood_evaluation,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,r2_gpt2_small_directed_pairs,test,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,66,66
2,gpt2-small,imdb,frozen_ood_evaluation,kokolamba/sentiment-manifold-imdb-pythia-2.8b,gpt2_small_directed_pairs,test,06586d20342aa46fa61525b1f7609ab065983e3e,06586d20342aa46fa61525b1f7609ab065983e3e,15274,15274
3,gpt2-small,sst,frozen_ood_evaluation,kokolamba/sentiment-manifold-sst-pythia-2.8b,tigges_gpt2_small_directed_pairs,test,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,304,304
4,qwen-0.6b,dynasent_r1,frozen_ood_evaluation,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,r1_qwen_0_6b_directed_pairs,test,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,80,80
5,qwen-0.6b,dynasent_r2,frozen_ood_evaluation,kokolamba/sentiment-manifold-dynasent-r1-r2-py...,r2_qwen_0_6b_directed_pairs,test,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,f4aa6cc90bfeda2080e6445592a8f689c974fd3d,68,68
6,qwen-0.6b,imdb,frozen_ood_evaluation,kokolamba/sentiment-manifold-imdb-pythia-2.8b,qwen_0_6b_directed_pairs,test,06586d20342aa46fa61525b1f7609ab065983e3e,06586d20342aa46fa61525b1f7609ab065983e3e,15218,15218
7,qwen-0.6b,sst,frozen_ood_evaluation,kokolamba/sentiment-manifold-sst-pythia-2.8b,tigges_qwen_0_6b_directed_pairs,test,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,bcffb933a34a48b409a7caf3a53f0fe7bb8152cc,302,302


Validated 16 aggregate metric rows and all per-case records.


In [8]:
from IPython.display import Markdown

DATASET_LABELS = {
    "sst": "SST", "imdb": "IMDb",
    "dynasent_r1": "DynaSent R1", "dynasent_r2": "DynaSent R2",
}
TABLE_METRICS = {
    "logit_flip_percent": "Logit Flip Percent",
    "sign_flip_percent": "Literal Sign Flip Percent",
}
for model_name in RESULT_MODEL_NAMES:
    model_metrics = metrics[metrics["model"] == model_name].copy()
    for metric_column, metric_title in TABLE_METRICS.items():
        if model_metrics[metric_column].isna().any():
            raise RuntimeError(f"Missing {metric_column} values for {model_name}.")
        table = model_metrics.pivot(
            index="method", columns="dataset", values=metric_column
        ).reindex(index=METHOD_ORDER, columns=DATASET_ORDER)
        table.index = [METHOD_LABELS[method] for method in METHOD_ORDER]
        table.columns = [DATASET_LABELS[dataset] for dataset in DATASET_ORDER]
        display(Markdown(f"### {MODEL_LABELS[model_name]} — {metric_title}"))
        display(table.style.format("{:.1f}").set_properties(**{"text-align": "center"}))

### GPT-2 Small — Logit Flip Percent

,SST,IMDb,DynaSent R1,DynaSent R2
Mean Difference,39.2,39.2,77.8,46.9
DAS (1D),101.7,116.5,161.1,109.4


### GPT-2 Small — Literal Sign Flip Percent

,SST,IMDb,DynaSent R1,DynaSent R2
Mean Difference,0.0,0.4,0.0,0.0
DAS (1D),4.6,17.3,2.5,6.1


### Qwen3-0.6B Base — Logit Flip Percent

,SST,IMDb,DynaSent R1,DynaSent R2
Mean Difference,89.6,88.8,92.2,91.2
DAS (1D),103.6,101.7,101.6,100.0


### Qwen3-0.6B Base — Literal Sign Flip Percent

,SST,IMDb,DynaSent R1,DynaSent R2
Mean Difference,79.1,82.6,72.5,79.4
DAS (1D),93.7,96.3,82.5,95.6


## 7. Final paths and credential cleanup

In [9]:
clear_hf_credentials(config.hf_token_env)
assert config.hf_token_env not in os.environ
assert "HF_TOKEN" not in _RUNTIME_SECRETS
assert "_token" not in globals()
print("Verified: HF_TOKEN is absent from the environment and notebook secret cache.")
print("Completed run:", RUN_LAYOUT.root)
print("Manifest:     ", RUN_LAYOUT.manifest_path)
print("Combined CSVs:", RUN_LAYOUT.results_dir)
for model_name in RESULT_MODEL_NAMES:
    print("Per-model:   ", RUN_LAYOUT.results_dir / model_name)

Verified: HF_TOKEN is absent from the environment and notebook secret cache.
Completed run: /content/drive/MyDrive/sentiment-geometry/ait-valence-ood-evaluation/runs/2026-09-24_11-20_CDT
Manifest:      /content/drive/MyDrive/sentiment-geometry/ait-valence-ood-evaluation/runs/2026-09-24_11-20_CDT/run_manifest.json
Combined CSVs: /content/drive/MyDrive/sentiment-geometry/ait-valence-ood-evaluation/runs/2026-09-24_11-20_CDT/results
Per-model:    /content/drive/MyDrive/sentiment-geometry/ait-valence-ood-evaluation/runs/2026-09-24_11-20_CDT/results/gpt2-small
Per-model:    /content/drive/MyDrive/sentiment-geometry/ait-valence-ood-evaluation/runs/2026-09-24_11-20_CDT/results/qwen-0.6b


## Interpretation boundary

These are locked cross-dataset transfer results for valence directions learned on full AIT training data and selected on AIT validation. SST, IMDb, DynaSent R1, and DynaSent R2 are confirmation/OOD datasets only: their results must not be used to reselect a layer, DAS checkpoint, method, patch position, direction sign, or intervention strength.

In [10]:
import gc
import sys
import torch

# Release references retained by the most recent exception traceback.
sys.last_traceback = None
sys.last_value = None
sys.last_type = None

gc.collect()

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    free_gb, total_gb = torch.cuda.mem_get_info()
    allocated_gb = torch.cuda.memory_allocated() / 2**30
    reserved_gb = torch.cuda.memory_reserved() / 2**30

    print(f"Free:      {free_gb:.2f} GB")
    print(f"Allocated: {allocated_gb:.2f} GB")
    print(f"Reserved:  {reserved_gb:.2f} GB")
    print(f"Total:     {total_gb / 2**30:.2f} GB")

Free:      41835495424.00 GB
Allocated: 0.01 GB
Reserved:  0.02 GB
Total:     39.49 GB
